![Week 2 Day 2](../assets/banners/w2d2.png)

# Week 2 Day 2 - Loading PDFs and Chunking Them Well

Yesterday you saw why GradeWise needs RAG, and you turned words into vectors by hand. You also did the "retrieval" yourself, by copying the marking-principles page out of the handbook.

Today we automate the first half of that. Asha's notes are PDFs, and PDFs are a *layout* format, not a *text* format: they know where each glyph goes on the page, not which line is a heading or where a section ends. So we'll hold a **loader bake-off** on the real Unit 1 notes, then cut the text into **chunks**, comparing four strategies side by side on a picture of the document.

Get this step wrong and nothing downstream can save you: a retriever can only return the chunks you gave it.

![Your 6-week roadmap](../assets/roadmap/week2.png)

### 🎯 By the end of this lab you will
1. 📄 Compare raw PyMuPDF text with pymupdf4llm Markdown on the same page, and know when PyPDF or Docling are worth it
2. 🧹 Strip running headers, footers and page numbers that would pollute every chunk
3. ✂️ Compare fixed, recursive, semantic and structure-aware chunking, visually, on the real Unit 1 PDF
4. 🏷️ Produce GradeWise's chunks: each one with its source, page, unit and section path

### 🎬 Videos for this lab

| # | Lecture |
|:-:|---|
| 1 | Day 2 - Chunking Strategies: Fixed, Recursive, Semantic, Structure-Aware |
| 2 | Day 2 - Loading PDFs: PyPDF vs PyMuPDF vs pymupdf4llm vs Docling |
| 3 | Day 2 - Structure-Aware Chunking with Rich Metadata |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">This whole lab is <b>offline and free</b>: no model is called. We work on the real file <code>data/knowledge-base/unit1_operating_systems.pdf</code> (7 pages). Want to see what the PDF was made from? The source is <code>scripts/knowledge_base_source/unit1_operating_systems.md</code>.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pymupdf
import pymupdf4llm

from gradewise.config import DATA_DIR

UNIT1 = DATA_DIR / "knowledge-base" / "unit1_operating_systems.pdf"
with pymupdf.open(UNIT1) as book:
    print(f"{UNIT1.name}: {len(book)} pages")
    print("PDF metadata:", {k: book.metadata[k] for k in ("title", "author", "keywords")})

## Part 1: The loader bake-off

There are four loaders worth knowing in Python. We have two installed, which are the two GradeWise chooses between:

| Loader | Gives you | Speed | Good for |
|---|---|---|---|
| **PyPDF** (`pypdf`) | Plain text, pure Python | 🐢 | Simple PDFs, no compiled dependencies |
| **PyMuPDF** (`pymupdf`) | Plain text, plus positions, fonts, images | 🚀 | Fast raw text; the engine under the next one |
| **pymupdf4llm** | **Markdown**: `#` headings, `**bold**`, lists, tables | 🚀 | ⭐ *Our choice*: structure survives, and it's still fast |
| **Docling** (IBM) | Markdown/JSON from a layout + table AI model, OCR | 🐌 (downloads models) | Scanned PDFs, complex tables, multi-column papers |

Let's put the two installed ones head to head on page 4, where section 1.5 ends and **1.6 Deadlocks** begins.

In [ ]:
PAGE = 4  # 💡 1-based, the way a human reads page numbers

t0 = time.perf_counter()
with pymupdf.open(UNIT1) as book:
    raw_pages = [page.get_text() for page in book]
raw_seconds = time.perf_counter() - t0

print(f"⏱️ PyMuPDF raw text: {raw_seconds * 1000:.0f} ms for {len(raw_pages)} pages\n")
print(raw_pages[PAGE - 1][-1400:])

In [ ]:
t0 = time.perf_counter()
md_pages = pymupdf4llm.to_markdown(str(UNIT1), page_chunks=True, show_progress=False)
md_seconds = time.perf_counter() - t0

print(f"⏱️ pymupdf4llm Markdown: {md_seconds * 1000:.0f} ms for {len(md_pages)} pages\n")
print(md_pages[PAGE - 1]["text"][-1600:])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Compare the two outputs before reading on. In the raw text, can you tell that "1.6 Deadlocks" is a heading? Search both for the word <code>five</code> (as in "at most five threads"). Predict: will a plain <code>"five" in text</code> find it in both versions? Run the next cell.</span>
        </td>
    </tr>
</table>

In [ ]:
raw, mdtext = raw_pages[PAGE - 1], md_pages[PAGE - 1]["text"]

print(f"{'':<34}{'PyMuPDF raw':>14}{'pymupdf4llm':>14}")
ligatures = ("\ufb01", "\ufb00")  # 'ﬁ' and 'ﬀ'
checks = {
    "'five' found by plain search": ("five" in raw, "five" in mdtext),
    "'ﬁ' ligature characters": (sum(map(raw.count, ligatures)), sum(map(mdtext.count, ligatures))),
    "Markdown '## ' headings": (raw.count("\n## "), mdtext.count("\n## ") + mdtext.startswith("## ")),
    "**bold** spans": (raw.count("**") // 2, mdtext.count("**") // 2),
    "characters": (len(raw), len(mdtext)),
}
for name, (a, b) in checks.items():
    print(f"{name:<34}{str(a):>14}{str(b):>14}")

Two lessons in one table:

1. **Ligatures.** PDFs often store "fi", "ff" and "fl" as single glyphs (`ﬁ`, `ﬀ`). Raw text keeps them, so the word *five* is really `ﬁve`, and a keyword search (or BM25 on Day 4) for "five" or "buffer" silently misses it. pymupdf4llm normalises them.
2. **Structure.** Raw text flattens "1.6 Deadlocks" into just another line. Markdown keeps it as `## **1.6 Deadlocks**`, and that heading is what lets us chunk by section later today.

What about the other two loaders? We don't install them (Docling pulls in large AI models), but here is what they'd look like, so you can try them in your own projects:

```python
# PyPDF: uv add pypdf
from pypdf import PdfReader
text = PdfReader("unit1_operating_systems.pdf").pages[3].extract_text()

# Docling: uv add docling   (first run downloads layout + table models, ~1 GB)
from docling.document_converter import DocumentConverter
result = DocumentConverter().convert("unit1_operating_systems.pdf")
markdown = result.document.export_to_markdown()
```

💡 Rule of thumb: start with pymupdf4llm. Reach for Docling when your PDFs are **scanned** (it does OCR), have **complex tables**, or use **two columns**. Asha's notes are clean, born-digital PDFs, so the fast option wins.

### Running headers, footers and page numbers

Look at the top of any page from pymupdf4llm: every page starts with a running header, and the PDF also has a footer. If we left them in, *every chunk* would contain "CS201 Systems Fundamentals · Northbridge Institute" and "Page 4 of 7", which is noise for embeddings and BM25 alike.

In [ ]:
for page in md_pages[:3]:
    lines = [ln for ln in page["text"].splitlines() if ln.strip()]
    print(f"page {page['metadata']['page']}:  first line → {lines[0][:60]!r}")
    print(f"         last line  → {lines[-1][:60]!r}")

GradeWise's loader finds those lines automatically: lines near the top or bottom of a page that repeat (with digits blanked out, so "Page 3 of 7" and "Page 4 of 7" match) on most pages. Here are the two helpers from `gradewise/ingestion/loaders.py` at work:

In [ ]:
from gradewise.ingestion.loaders import find_running_lines, strip_running_lines

texts = [p["text"] for p in md_pages]
running = find_running_lines(texts)
print("Detected running lines (digits shown as #):")
for sig in sorted(running):
    print("  •", sig)

before = texts[PAGE - 1]
after = strip_running_lines(before, running)
print(f"\nPage {PAGE}: {len(before)} characters before cleaning, {len(after)} after")

`load_pdf()` puts all of this together: pymupdf4llm → strip running lines → one LangChain `Document` per page, with the metadata every later stage relies on.

In [ ]:
from gradewise.ingestion import load_pdf

pages = load_pdf(UNIT1)
print(f"{len(pages)} page Documents\n")
for doc in pages:
    print(doc.metadata)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Two details in that metadata will save you pain later. <code>page</code> is <b>1-based</b>, like the number a human reads at the bottom of the page, so a citation "p4" means what Asha thinks it means. And <code>section</code> is the section <i>in force where the page starts</i>: page 2 opens in the middle of "1.2 Processes", which began on page 1. Without carrying that forward, the first paragraph of every page would belong to no section at all.</span>
        </td>
    </tr>
</table>

## Part 2: Four chunking strategies

Why chunk at all? Three reasons: embedding models have input limits; a vector for a whole 7-page unit is a blurry average of everything in it; and we want to paste only the *relevant* paragraphs into the prompt, not the whole book.

| Strategy | How it cuts | Strength | Weakness |
|---|---|---|---|
| **Fixed-size** | Every *N* characters, no matter what | Dead simple, predictable | Cuts mid-word, mid-sentence, mid-topic |
| **Recursive** | Tries `\n\n`, then `\n`, then `. `, then spaces, to stay under *N* | Respects paragraphs and sentences | Still blind to topics: a chunk can straddle two sections |
| **Semantic** | Embeds each sentence; cuts where neighbouring sentences stop being similar | Follows topic shifts | Costs an embedding per sentence; noisy on short sentences |
| **Structure-aware** | Splits on the document's own headings first, then recursively inside long sections | Chunks = sections, and the heading becomes metadata | Needs a loader that keeps headings (hello, pymupdf4llm) |

To compare them fairly, we'll chunk the same text: the cleaned Unit 1 notes, joined into one string. We also record where each `##` section starts, so we can *see* which chunks cross a section boundary.

In [ ]:
full_text = "\n\n".join(doc.page_content for doc in pages)
section_starts = [m.start() for m in re.finditer(r"^## ", full_text, flags=re.MULTILINE)]
section_names = [
    full_text[s : full_text.index("\n", s)].replace("#", "").replace("*", "").strip() for s in section_starts
]

print(f"{len(full_text):,} characters, {len(section_starts)} sections:")
for start, name in zip(section_starts, section_names, strict=True):
    print(f"  @{start:>6,}  {name}")

### Strategy 1: fixed-size

The bluntest possible approach: slice every 500 characters.

In [ ]:
SIZE = 500

fixed = [(i, min(i + SIZE, len(full_text))) for i in range(0, len(full_text), SIZE)]
print(f"{len(fixed)} fixed-size chunks. Chunk 7 begins and ends like this:\n")
start, end = fixed[7]
print(repr(full_text[start : start + 80]), "…", repr(full_text[end - 80 : end]))

### Strategy 2: recursive

LangChain's `RecursiveCharacterTextSplitter` tries the "nicest" separator first and falls back only when a piece is still too big. `add_start_index=True` records where each chunk starts, so we can draw it.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=SIZE, chunk_overlap=0, add_start_index=True)
recursive_docs = recursive_splitter.create_documents([full_text])
recursive = [(d.metadata["start_index"], d.metadata["start_index"] + len(d.page_content)) for d in recursive_docs]

print(f"{len(recursive)} recursive chunks. Chunk 7 begins and ends like this:\n")
print(repr(recursive_docs[7].page_content[:80]), "…", repr(recursive_docs[7].page_content[-80:]))

### Strategy 3: semantic

Semantic chunking embeds every sentence and cuts where two neighbouring sentences are unusually *dissimilar*. Real implementations use an embedding model (for example `SemanticChunker` in `langchain_experimental`). To stay offline, we'll use **TF-IDF vectors** from scikit-learn as a stand-in: they only capture shared words, not meaning, but they're real (not random) and good enough to show the idea.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 💡 Sentence boundaries, with their positions in full_text
SENTENCE = re.compile(r"[^.!?\n]+(?:[.!?]+|\n|$)")
sentence_spans = [(m.start(), m.end()) for m in SENTENCE.finditer(full_text) if m.group().strip()]
sentences = [full_text[s:e] for s, e in sentence_spans]

tfidf = TfidfVectorizer(stop_words="english").fit_transform(sentences).toarray()

# 💡 Compare a small WINDOW of sentences either side of each gap (real semantic chunkers do the
#    same, with a "buffer"): single short sentences often share no words with their neighbour.
WINDOW = 3


def window_similarity(gap: int) -> float:
    before = tfidf[max(0, gap - WINDOW + 1) : gap + 1].sum(axis=0)
    after = tfidf[gap + 1 : gap + 1 + WINDOW].sum(axis=0)
    norm = np.linalg.norm(before) * np.linalg.norm(after)
    return float(before @ after / norm) if norm else 0.0


adjacent = np.array([window_similarity(i) for i in range(len(sentences) - 1)])

# Cut where similarity drops below the 20th percentile, keeping chunks between ~200 and ~1,000 characters
threshold = np.percentile(adjacent, 20)
semantic, chunk_start = [], 0
for i, sim in enumerate(adjacent):
    end = sentence_spans[i][1]
    if (sim <= threshold and end - chunk_start > 200) or end - chunk_start > 1000:
        semantic.append((chunk_start, end))
        chunk_start = sentence_spans[i + 1][0]
semantic.append((chunk_start, len(full_text)))
# 💡 With TF-IDF the threshold is often 0.00: "cut where the two windows share no words at all"
print(f"{len(sentences)} sentences → {len(semantic)} semantic chunks (cut where similarity ≤ {threshold:.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
midpoints = [sentence_spans[i][1] for i in range(len(adjacent))]
ax.plot(midpoints, adjacent, color="#7c5cff", lw=1)
ax.axhline(threshold, color="#ef4444", ls="--", lw=1, label="cut threshold")
for s in section_starts:
    ax.axvline(s, color="#22c55e", alpha=0.6, lw=1)
ax.set_xlabel("position in Unit 1 (characters)   ·   green lines = real ## section starts")
ax.set_ylabel("similarity across the gap")
ax.set_title("Semantic chunking: cut where neighbouring sentences stop sharing vocabulary")
ax.legend(loc="upper right")
plt.show()

### Strategy 4: structure-aware (GradeWise's choice)

The notes already tell us where topics begin: their headings. LangChain's `MarkdownHeaderTextSplitter` splits on them and records the heading path as metadata; GradeWise then cuts long sections with the recursive splitter. Here is the idea on one page:

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

header_splitter = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2"), ("###", "h3")], strip_headers=False)
for piece in header_splitter.split_text(pages[PAGE - 1].page_content):
    print(f"{piece.metadata}  →  {len(piece.page_content)} chars: {piece.page_content[:50]!r}…")

The page splits into the tail of 1.5 (no heading on this page: that's why the loader carries the section forward) and the start of 1.6. Now the real thing, `chunk_documents()` from `gradewise/ingestion/chunking.py`. We ask it for 500-character chunks so the comparison with the other strategies is fair:

In [ ]:
from gradewise.ingestion import chunk_documents

structured_chunks = chunk_documents(pages, chunk_size=SIZE, chunk_overlap=0, contextual_header=False)


def locate(chunks_text: list[str]) -> list[tuple[int, int]]:
    """Find where each chunk sits in full_text, by searching for its first and last lines.

    💡 The header splitter re-joins lines (it adds two spaces before each line break), so we
       can't search for the whole chunk, only for pieces of text inside a single line.
    """
    spans, cursor = [], 0
    for text in chunks_text:
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        head, tail = lines[0][:40], lines[-1][-40:]
        start = full_text.find(head, cursor)
        start = start if start >= 0 else cursor
        end = full_text.find(tail, start)
        end = end + len(tail) if end >= 0 else start + len(text)
        spans.append((start, end))
        cursor = start + 1
    return spans


structured = locate([c.page_content for c in structured_chunks])
print(f"{len(structured)} structure-aware chunks")

### The visual comparison

One row per strategy. Each coloured block is a chunk; the green lines are the real section starts. A chunk that **crosses** a green line mixes two topics, and is drawn in red.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before you run it, predict the number of section-crossing chunks for each strategy. Which one do you expect to be zero? Will semantic chunking, which never looks at headings, still find most of the boundaries?</span>
        </td>
    </tr>
</table>

In [ ]:
def crosses(span: tuple[int, int]) -> bool:
    start, end = span
    return any(start + 5 < s < end - 5 for s in section_starts)


strategies = {"Fixed-size": fixed, "Recursive": recursive, "Semantic (TF-IDF)": semantic, "Structure-aware": structured}

fig, ax = plt.subplots(figsize=(12, 3.6))
for row, spans in enumerate(strategies.values()):
    for i, span in enumerate(spans):
        colour = "#ef4444" if crosses(span) else ("#7c5cff" if i % 2 else "#22d3ee")
        ax.barh(row, span[1] - span[0], left=span[0], height=0.6, color=colour, edgecolor="white", lw=0.5)
for s in section_starts:
    ax.axvline(s, color="#22c55e", lw=1.5)
ax.set_yticks(range(len(strategies)), [f"{n} ({len(s)})" for n, s in strategies.items()])
ax.invert_yaxis()
ax.set_xlabel("position in Unit 1 (characters)")
ax.set_title("Four ways to chunk Unit 1 · red = a chunk that mixes two sections")
plt.show()

for name, spans in strategies.items():
    print(f"{name:<20} {len(spans):>3} chunks, {sum(map(crosses, spans)):>2} cross a section boundary")

Fixed-size and recursive chunks happily straddle sections: one of them might hold the end of *1.5 Synchronisation* and the start of *1.6 Deadlocks*, and its vector becomes a blur of both. Semantic chunking catches some boundaries from word changes alone. Structure-aware chunking never crosses one, by construction.

## Part 3: Structure-aware chunking with rich metadata

Now the real settings: `chunk_documents(pages)` with its defaults (`CHUNK_SIZE = 1000`, `CHUNK_OVERLAP = 150`, `contextual_header=True`, `min_chunk_size=200`).

💡 That last one is a fix from testing: the recursive splitter sometimes left a tiny leftover at the end of a section (an early version produced a chunk that was just the word *storage.*). Now any chunk under 200 characters is merged into a neighbour **from the same section**, so it can never mix topics; a genuinely short section with no neighbour stays as it is.

In [ ]:
chunks = chunk_documents(pages)
deadlock_chunk = next(c for c in chunks if c.metadata["section"] == "1.6 Deadlocks")

print(f"{len(chunks)} chunks from {len(pages)} pages\n")
print(deadlock_chunk.metadata, "\n")
print(deadlock_chunk.page_content[:500])

Every chunk now carries:

| Field | Example | Used for |
|---|---|---|
| `source` | `unit1_operating_systems.pdf` | Citations; checking the right PDF was found (Day 5's Source@k) |
| `page` | `4` | Citations a human can check |
| `title`, `unit` | `CS201 Unit 1: Operating Systems`, `1` | Metadata filters: "only search Unit 1" (Day 3) |
| `section`, `section_path` | `1.6 Deadlocks` | Citations like `§1.6 Deadlocks`; tells the grader *which topic* the evidence is from |
| `chunk_index` | `9` | A stable ID, so re-ingesting never duplicates (Day 3) |

And look at the first line of the chunk: `[CS201 Unit 1: Operating Systems > 1.6 Deadlocks]`. That's a **contextual chunk header**. A chunk that says "*all four of these conditions must hold*" doesn't contain the word *deadlock*; prepending the section title fixes that for both embeddings and keyword search.

In [ ]:
from collections import Counter

lengths = [len(c.page_content) for c in chunks]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=12, color="#7c5cff", edgecolor="white")
ax.axvline(1000, color="#ef4444", ls="--", label="chunk_size = 1000")
ax.set_xlabel("characters per chunk")
ax.set_ylabel("chunks")
ax.set_title("Unit 1 chunk sizes: short ones are short sections, not mistakes")
ax.legend()
plt.show()

for section, n in Counter(c.metadata["section"] for c in chunks).items():
    print(f"{n} × {section}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Chunking is where most real-world RAG projects quietly win or lose. Teams often spend weeks swapping embedding models and vector databases, when the real problem is that their chunks cut a contract clause, a policy rule or a table in half. Preserving structure and metadata is cheap to do once, and it pays off in every answer, every citation and every audit afterwards.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Open <code>gradewise/ingestion/loaders.py</code> (<code>load_pdf</code>, <code>find_running_lines</code>, <code>strip_running_lines</code>) and <code>gradewise/ingestion/chunking.py</code> (<code>chunk_documents</code>, <code>CHUNK_SIZE</code>, <code>CHUNK_OVERLAP</code>). Everything you did today is in those two files, with the pitfalls marked ⚠️. In the week 2 exercise you'll tune <code>CHUNK_SIZE</code> and measure the effect.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 📄 ran a loader bake-off: raw PyMuPDF text loses headings and keeps ligatures; pymupdf4llm gives clean Markdown
- 🧹 detected and stripped running headers, footers and page numbers automatically
- ✂️ compared fixed, recursive, semantic and structure-aware chunking, and *saw* which ones mix topics
- 🏷️ produced GradeWise's chunks, each with source, page, unit, section path and a contextual header

**Tomorrow (Day 3):** these chunks need a home. We'll put them in a persistent Chroma vector store, filter by metadata, and make ingestion **idempotent**: run it twice and the second run adds 0 chunks and costs $0.

⏭ **Next:** `week2/day3.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Chunk <code>unit3_databases.pdf</code> instead. (1) For chunk sizes 300, 1000 and 2000, count the chunks and how many cross a section boundary, for both the recursive splitter and <code>chunk_documents</code>. (2) Find the shortest chunk <code>chunk_documents</code> produces: is it a problem, or a genuinely short section? (3) Turn <code>contextual_header</code> off and find a chunk that no longer mentions its topic. Solutions are in <code>week2/solutions/day2_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• pymupdf4llm: <a href="https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/">pymupdf.readthedocs.io/en/latest/pymupdf4llm</a><br/> • Docling: <a href="https://github.com/docling-project/docling">github.com/docling-project/docling</a><br/> • LangChain text splitters: <a href="https://docs.langchain.com/oss/python/integrations/splitters">docs.langchain.com/oss/python/integrations/splitters</a></span>
        </td>
    </tr>
</table>